[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bulentsoykan/ABM-particle-filter-notebooks/blob/main/notebook_03_pf_abm_spatial.ipynb)

# Module 3: Applying the Particle Filter to an Agent-Based Model

**Learning objectives**
1. Understand the "identical twin" experimental design
2. See each particle as a *full ABM simulation* — not just a number
3. Visualize how the PF constrains the ensemble towards the pseudo-truth
4. Observe that more agents require more particles

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.alpha': 0.5,  'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'axes.grid': True,
    'font.size': 11,
})
os.makedirs('figures', exist_ok=True)
print("Libraries loaded.")

In [ ]:
# ── Minimal StationSim-style corridor ABM ──────────────────────────────────────

class Agent:
    '''A pedestrian moving left to right through a corridor.

    The ONLY source of randomness: when a faster agent catches up to a slower one,
    it randomly dodges LEFT or RIGHT (50/50 coin flip).  Everything else is
    deterministic.  That one binary choice is all it takes to cause divergence.
    '''
    def __init__(self, agent_id, env_width=100, env_height=50, rng=None):
        rng = rng or np.random
        self.id         = agent_id
        self.x          = rng.uniform(0, 3)
        self.y          = rng.uniform(5, env_height - 5)
        self.max_speed  = max(0.3, rng.normal(1.0, 0.4))
        self.env_width  = env_width
        self.env_height = env_height
        self.active     = True
        self.traj_x     = [self.x]
        self.traj_y     = [self.y]

    def step(self, all_agents, separation=4.5, rng=None):
        rng = rng or np.random
        if not self.active:
            self.traj_x.append(self.x)
            self.traj_y.append(self.y)
            return
        speed, dy = self.max_speed, 0.0
        for other in all_agents:
            if other.id == self.id or not other.active:
                continue
            dx   = other.x - self.x
            dist = np.hypot(dx, other.y - self.y)
            if 0 < dx < separation * 2 and dist < separation:
                speed *= 0.6
                dy = 1.0 if rng.random() < 0.5 else -1.0   # THE random choice
                break
        self.x = min(self.x + speed, self.env_width)
        self.y = np.clip(self.y + dy, 2, self.env_height - 2)
        if self.x >= self.env_width:
            self.active = False
        self.traj_x.append(self.x)
        self.traj_y.append(self.y)


class CorridorABM:
    '''Corridor where N agents walk left-to-right.

    Simplified StationSim from Malleson et al. (2020).
    State vector: [x0,y0, x1,y1, ..., x_{N-1},y_{N-1}]  — dimension 2*N_agents.
    '''
    def __init__(self, n_agents=10, env_width=100, env_height=50, seed=None):
        rng             = np.random.RandomState(seed)
        self.n_agents   = n_agents
        self.env_width  = env_width
        self.env_height = env_height
        self.rng        = rng
        self.agents     = [Agent(i, env_width, env_height, rng)
                           for i in range(n_agents)]
        self.pos_history = [self.get_positions()]
        self.step_count  = 0
        self.collisions  = 0

    def get_positions(self):
        return np.array([[a.x, a.y] for a in self.agents])

    def get_state_vector(self):
        '''Flat [x0,y0, x1,y1,...] — used by the particle filter.'''
        return self.get_positions().flatten()

    def step(self):
        before = [a.active for a in self.agents]
        for agent in self.agents:
            agent.step(self.agents, rng=self.rng)
        # count collision events (agents that slowed down)
        for a in self.agents:
            for b in self.agents:
                if a.id >= b.id or not a.active or not b.active:
                    continue
                if np.hypot(a.x - b.x, a.y - b.y) < 4.5:
                    self.collisions += 1
        self.pos_history.append(self.get_positions())
        self.step_count += 1

    def run(self, n_steps):
        for _ in range(n_steps):
            self.step()
        return self

print("CorridorABM defined.  State dimension = 2 x N_agents.")


In [ ]:
def systematic_resample(weights):
    '''Systematic resampling — O(N), low variance, preferred for PF.

    Algorithm:
      1. Draw one random U ~ Uniform[0, 1/N]
      2. Space N points evenly: U, U+1/N, U+2/N, ..., U+(N-1)/N
      3. Walk the CDF; each particle covers a slice proportional to its weight
         -> heavy particles get sampled multiple times, light ones get dropped
    '''
    N   = len(weights)
    w   = np.asarray(weights, dtype=float)
    w   = w / w.sum()
    U   = np.random.uniform(0, 1.0 / N)
    pts = (np.arange(N) + U) / N
    cdf = np.cumsum(w)
    i, j, idxs = 0, 0, np.zeros(N, dtype=int)
    while i < N:
        if pts[i] <= cdf[j]:
            idxs[i] = j
            i += 1
        else:
            j = min(j + 1, N - 1)
    return idxs

from copy import deepcopy

class ParticleFilterABM:
    '''SIR Particle Filter applied to CorridorABM.

    Each 'particle' is a full ABM instance.  Every resample_window steps:
      PREDICT  -> advance all N_p ABMs one step
      REWEIGHT -> score each particle against the pseudo-truth observation
      RESAMPLE -> duplicate high-weight particles, discard low-weight ones
      JITTER   -> add small Gaussian noise to prevent particle deprivation

    '''
    def __init__(self, n_particles, n_agents, env_width=100, env_height=50,
                 particle_std=0.25, resample_window=20):
        self.n_p    = n_particles
        self.n_a    = n_agents
        self.pstd   = particle_std
        self.window = resample_window
        self.particles = [
            CorridorABM(n_agents, env_width, env_height, seed=i)
            for i in range(n_particles)
        ]
        self.weights     = np.ones(n_particles) / n_particles
        self.w_history   = []    # weights at each update (for degeneracy viz)
        self.err_history = []    # mean weighted error at each update
        self.step_count  = 0

    def predict(self):
        for p in self.particles:
            p.step()
        self.step_count += 1

    def update(self, observation):
        '''Reweight + resample against observed state vector.'''
        # Eq.(3): epsilon_i = ||y_k - x_k^i||_2
        errors = np.array([
            np.linalg.norm(p.get_state_vector() - observation)
            for p in self.particles
        ])
        # Weight: w_i = eta / (1e-9 + epsilon_i)
        self.weights = 1.0 / (1e-9 + errors)
        self.weights /= self.weights.sum()
        self.w_history.append(self.weights.copy())
        self.err_history.append(float(np.dot(self.weights, errors)))

        # Systematic resample
        idxs = systematic_resample(self.weights)
        self.particles = [deepcopy(self.particles[k]) for k in idxs]
        self.weights   = np.ones(self.n_p) / self.n_p

        # Jitter / roughening: prevents all particles collapsing to one state
        if self.pstd > 0:
            for p in self.particles:
                noise = np.random.randn(self.n_a, 2) * self.pstd
                for k, ag in enumerate(p.agents):
                    ag.x = np.clip(ag.x + noise[k, 0], 0, p.env_width)
                    ag.y = np.clip(ag.y + noise[k, 1], 2, p.env_height - 2)

    def get_estimate(self):
        '''Weighted mean state estimate (Eq. 7).'''
        states = np.array([p.get_state_vector() for p in self.particles])
        return np.average(states, weights=self.weights, axis=0)

    def best_particle(self):
        return self.particles[np.argmax(self.weights)]

    def run(self, truth_model, n_steps):
        '''Run PF alongside the truth model for n_steps.'''
        for t in range(n_steps):
            self.predict()
            truth_model.step()
            if (t + 1) % self.window == 0:
                self.update(truth_model.get_state_vector())
        return self

print("ParticleFilterABM defined.")


## The Identical Twin Experiment

We follow the experimental design from the paper (Section 1.6):

```
  Pseudo-truth model  ─────► noisy observations  ──────┐
       (seed=0)                 y_k = x_k + noise        │
                                                          ▼
  Particle 1 (seed=1) ──┐                          ┌──────────────┐
  Particle 2 (seed=2) ──┤ ── step forward ─────────► Reweight +   ──► New particles
  ...                   │                          │  Resample     │
  Particle N (seed=N) ──┘                          └──────────────┘
```

Because we *know* the pseudo-truth, we can compute the exact PF error.
In reality, the true state can never be known precisely.


In [ ]:
# ── Run a small example: 6 agents, 50 particles, 60 steps ────────────────────
np.random.seed(0)
N_A, N_P_SMALL, N_STEPS = 6, 50, 60
WINDOW = 20    # observe every 20 steps

truth_sm = CorridorABM(n_agents=N_A, seed=0)
pf_sm    = ParticleFilterABM(n_particles=N_P_SMALL, n_agents=N_A,
                              particle_std=0.25, resample_window=WINDOW)

snapshot_steps = [0, WINDOW, 2*WINDOW, 3*WINDOW]
snapshots = {}

for t in range(N_STEPS):
    pf_sm.predict()
    truth_sm.step()
    if (t + 1) % WINDOW == 0:
        pf_sm.update(truth_sm.get_state_vector())
    if t in snapshot_steps:
        snapshots[t] = {
            'truth': truth_sm.get_positions().copy(),
            'particles': [p.get_positions().copy() for p in pf_sm.particles],
            'weights': pf_sm.weights.copy(),
        }

print(f"Run complete: {N_A} agents, {N_P_SMALL} particles, {N_STEPS} steps.")
print(f"Data assimilation windows: {N_STEPS // WINDOW}")
print(f"PF errors at each window: {[f'{e:.2f}' for e in pf_sm.err_history]}")


In [ ]:
# ── Visualise: pseudo-truth (black squares) vs best particle (red circles) ─────

def plot_pf_state(ax, truth_pos, particle_positions, weights, title, step):
    '''Plot one snapshot of the PF state.'''
    # Best particle
    best_idx = np.argmax(weights)
    best_pos = particle_positions[best_idx]

    # All particles as faint dots
    for pos in particle_positions[::5]:   # every 5th for clarity
        ax.scatter(pos[:, 0], pos[:, 1], s=8, color='#3fb950', alpha=0.15, zorder=1)

    # Truth and best-particle estimate
    ax.scatter(truth_pos[:, 0], truth_pos[:, 1], s=120, color='white',
               marker='s', zorder=5, label='Pseudo-truth')
    ax.scatter(best_pos[:, 0], best_pos[:, 1], s=80, color='#f9826c',
               marker='o', zorder=4, label='Best particle')

    # Error lines
    for i in range(len(truth_pos)):
        ax.plot([truth_pos[i,0], best_pos[i,0]],
                [truth_pos[i,1], best_pos[i,1]],
                color='yellow', lw=1, alpha=0.6, zorder=3)

    err = np.linalg.norm(truth_pos - best_pos)
    ax.set(xlim=(-2, 105), ylim=(0, 50), title=f'{title}  (step {step}, err={err:.1f})',
           xlabel='X (m)', ylabel='Y (m)')


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'PF State Snapshots — {N_A} agents, {N_P_SMALL} particles', fontsize=14, fontweight='bold')

for ax, (t, snap) in zip(axes.flat, sorted(snapshots.items())):
    plot_pf_state(ax, snap['truth'], snap['particles'], snap['weights'],
                  'DA window' if t > 0 else 'Initial', t)
    if t == 0:
        ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('figures/fig_03a_pf_abm_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Three-scenario comparison: more agents = more particles needed ─────────────
scenarios = [
    {'label': '5 agents / 10 particles',  'n_a': 5,  'n_p': 10},
    {'label': '10 agents / 10 particles', 'n_a': 10, 'n_p': 10},
    {'label': '10 agents / 100 particles','n_a': 10, 'n_p': 100},
]
N_RUNS, N_S = 12, 60

results = {}
for sc in scenarios:
    errs = []
    for run in range(N_RUNS):
        truth_sc = CorridorABM(n_agents=sc['n_a'], seed=run)
        pf_sc    = ParticleFilterABM(n_particles=sc['n_p'], n_agents=sc['n_a'],
                                      particle_std=0.25, resample_window=20)
        pf_sc.run(truth_sc, N_S)
        errs.append(pf_sc.err_history)
    results[sc['label']] = np.array(errs)   # (N_RUNS, n_windows)
    print(f"  {sc['label']:35s}  median final error: "
          f"{np.median(results[sc['label']][:,-1]):.2f}")

print("\nKey insight: 10 agents / 10 particles has HIGH error.")
print("Adding more particles (100) dramatically reduces it.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle('Three Scenarios — Impact of N_agents and N_particles', fontsize=14, fontweight='bold')
palette = ['#3fb950', '#f0883e', '#58a6ff']

for ax, (label, errs), col in zip(axes, results.items(), palette):
    n_windows = errs.shape[1]
    windows   = np.arange(1, n_windows + 1)
    med = np.median(errs, axis=0)
    lo  = np.percentile(errs, 25, axis=0)
    hi  = np.percentile(errs, 75, axis=0)
    ax.fill_between(windows, lo, hi, alpha=0.25, color=col)
    ax.plot(windows, med, color=col, lw=2.5, marker='o', ms=6)
    ax.set(xlabel='Data assimilation window', ylabel='Mean PF error (m)',
           title=label)
    ax.text(0.05, 0.95, f'Median final\nerror: {med[-1]:.1f} m',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='#161b22', ec=col, alpha=0.9))

plt.tight_layout()
plt.savefig('figures/fig_03b_three_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()
print("Observation: doubling agents with same N_particles causes error to spike.")
print("Need ~10x more particles to compensate for 2x more agents.")

## Key Takeaways

| Scenario | N_agents | N_particles | Result |
|----------|----------|-------------|--------|
| Easy | 5 | 10 | All particles track well — few collisions |
| Hard | 10 | 10 | Particles diverge — too few hypotheses |
| Fixed | 10 | 100 | Error drops — enough hypotheses to cover the state space |

**The core insight**:

The number of particles needed to maintain low error grows **exponentially**
with the number of agents.

Why? Each collision is a random binary choice. With N agents there are
O(N^2) collision pairs. The particle filter must maintain at least one particle
that gets *all* those choices "right" — exponentially hard as N grows.

**Next: Module 4** — particle degeneracy: what happens when the filter fails.
